In [6]:
"""
read_mye.py
===========
Reads ONS Mid-Year Population Estimates (MYE5 sheet)
from the mye24tablesew.xlsx file and previews the structure.

Run from project root:
  python scripts/read_mye.py
"""

import pandas as pd

BASE = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"
MYE_PATH = f"{BASE}/data/raw/population/mye24tablesew.xlsx"

# ── Read MYE5 sheet ────────────────────────────────────────────────────────
print("Reading MYE5 sheet...")

# Read without header first to see raw structure
raw = pd.read_excel(MYE_PATH, sheet_name="MYE5", header=None)

print(f"\nShape: {raw.shape}")
print(f"\nFirst 10 rows, first 10 columns:")
print(raw.iloc[:10, :10].to_string())

print(f"\nRow 0 values (likely title/notes):")
print(raw.iloc[0, :10].tolist())

print(f"\nRow 4 values (likely header row):")
print(raw.iloc[4, :10].tolist())

print(f"\nRow 5 values (likely first data row):")
print(raw.iloc[5, :10].tolist())

print(f"\nUnique values in column 0 (first 20):")
print(raw.iloc[:, 0].dropna().unique()[:20].tolist())

Reading MYE5 sheet...

Shape: (365, 32)

First 10 rows, first 10 columns:
                                                                                                               0                  1          2             3                              4                       5                              6                       7                              8                       9
0                      MYE5: Population density for local authorities in England and Wales, mid-2011 to mid-2024                NaN        NaN           NaN                            NaN                     NaN                            NaN                     NaN                            NaN                     NaN
1                                                This worksheet contains one table. Freeze panes are turned on.                 NaN        NaN           NaN                            NaN                     NaN                            NaN                     NaN                

In [7]:
"""
msa_population_ranks.py
=======================
Reads ONS MYE5, aggregates LA populations to MSA level,
computes England benchmarks, and ranks all 20 MSAs.

Run from project root:
  python scripts/msa_population_ranks.py
"""

import pandas as pd
import numpy as np

BASE     = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"
MYE_PATH = f"{BASE}/data/raw/population/mye24tablesew.xlsx"

# ── 1. Read MYE5 with correct header ──────────────────────────────────────
print("── 1. Reading MYE5 ──────────────────────────────────────────────")
df = pd.read_excel(MYE_PATH, sheet_name="MYE5", header=7)
df.columns = df.columns.str.strip()

# Rename for clarity
df = df.rename(columns={
    "Code":                           "la_code",
    "Name":                           "la_name",
    "Geography":                      "geography",
    "Area (sq km)":                   "area_km2",
    "Estimated Population mid-2024":  "pop_2024",
    "2024 people per sq. km":         "density_2024",
    "Estimated Population mid-2023":  "pop_2023",
    "2023 people per sq. km":         "density_2023",
    "Estimated Population mid-2022":  "pop_2022",
    "2022 people per sq. km":         "density_2022",
    "Estimated Population mid-2021":  "pop_2021",
    "2021 people per sq. km":         "density_2021",
    "Estimated Population mid-2020":  "pop_2020",
    "2020 people per sq. km":         "density_2020",
    "Estimated Population mid-2019":  "pop_2019",
    "2019 people per sq. km":         "density_2019",
    "Estimated Population mid-2018":  "pop_2018",
    "2018 people per sq. km":         "density_2018",
    "Estimated Population mid-2017":  "pop_2017",
    "2017 people per sq. km":         "density_2017",
    "Estimated Population mid-2016":  "pop_2016",
    "2016 people per sq. km":         "density_2016",
    "Estimated Population mid-2015":  "pop_2015",
    "2015 people per sq. km":         "density_2015",
    "Estimated Population mid-2014":  "pop_2014",
    "2014 people per sq. km":         "density_2014",
    "Estimated Population mid-2013":  "pop_2013",
    "2013 people per sq. km":         "density_2013",
    "Estimated Population mid-2012":  "pop_2012",
    "2012 people per sq. km":         "density_2012",
    "Estimated Population mid-2011":  "pop_2011",
    "2011 people per sq. km":         "density_2011",
})

# Drop rows with no LA code or non-LA geographies
df = df.dropna(subset=["la_code"])
df = df[df["la_code"].astype(str).str.match(r"^[EW]\d+")]

# Keep only district-level LAs (E06, E07, E08, E09 prefixes) + England
la_df = df[df["la_code"].str.startswith(("E06","E07","E08","E09","W06"))].copy()
england_row = df[df["la_code"] == "E92000001"].iloc[0]

print(f"   LA rows loaded: {len(la_df)}")
print(f"   Columns: {list(la_df.columns[:8])}")
print(f"   England population 2024: {england_row['pop_2024']:,.0f}")

# ── 2. MSA constituent LA lookup ──────────────────────────────────────────
print("\n── 2. Defining MSA boundaries ───────────────────────────────────")

MSA_LAS = {
    "Greater Manchester":          ["E08000001","E08000002","E08000003","E08000004",
                                    "E08000005","E08000006","E08000007","E08000008",
                                    "E08000009","E08000010"],
    "West Midlands":               ["E08000025","E08000026","E08000027","E08000028",
                                    "E08000029","E08000030","E08000031"],
    "North East":                  ["E06000047","E06000005","E06000001","E06000002",
                                    "E06000057","E06000003","E06000004",
                                    "E08000037","E08000021","E08000022",
                                    "E08000023","E08000024"],
    "Liverpool City Region":       ["E08000011","E08000012","E08000014",
                                    "E08000013","E08000015","E06000006"],
    "South Yorkshire":             ["E08000016","E08000017","E08000018","E08000019"],
    "West Yorkshire":              ["E08000032","E08000033","E08000034",
                                    "E08000035","E08000036"],
    "West of England":             ["E06000025","E06000022","E06000023","E06000024"],
    "Lancashire":                  ["E07000117","E07000118","E07000119","E07000120",
                                    "E07000121","E07000122","E07000123","E07000124",
                                    "E07000125","E07000126","E07000127","E07000128",
                                    "E06000008","E06000009"],
    "Tees Valley":                 ["E06000001","E06000002","E06000003",
                                    "E06000004","E06000005"],
    "York & North Yorkshire":      ["E06000014","E06000065"],
    "Hull & East Yorkshire":       ["E06000010","E06000011"],
    "Greater Lincolnshire":        ["E06000012","E06000013","E07000136","E07000137",
                                    "E07000138","E07000139","E07000140","E07000141",
                                    "E07000142"],
    "East Midlands":               ["E06000015","E06000016","E06000017","E06000018",
                                    "E07000129","E07000130","E07000131","E07000132",
                                    "E07000133","E07000134","E07000135",
                                    "E07000170","E07000171","E07000172","E07000173",
                                    "E07000174","E07000175","E07000176",
                                    "E07000177","E07000178","E07000179","E07000180",
                                    "E07000181"],
    "Cambridgeshire & Peterborough":["E06000031","E07000008","E07000009",
                                    "E07000010","E07000011","E07000012"],
    "Sussex & Brighton":           ["E07000061","E07000062","E07000063",
                                    "E07000064","E07000065",
                                    "E07000223","E07000224","E07000225",
                                    "E07000226","E07000227","E07000228","E07000229",
                                    "E06000043"],
    "Hampshire & Solent":          ["E07000084","E07000085","E07000086","E07000087",
                                    "E07000088","E07000089","E07000090","E07000091",
                                    "E07000092","E07000093","E07000094",
                                    "E06000044","E06000045"],
    "Cheshire & Warrington":       ["E06000049","E06000050","E06000007"],
    "Cumbria":                     ["E06000063","E06000064"],
    "Greater Essex":               ["E07000066","E07000067","E07000068","E07000069",
                                    "E07000070","E07000071","E07000072","E07000073",
                                    "E07000074","E07000075","E07000076","E07000077",
                                    "E06000033","E06000034"],
    "Norfolk & Suffolk":           ["E07000143","E07000144","E07000145","E07000146",
                                    "E07000147","E07000148","E07000149",
                                    "E07000200","E07000202","E07000203",
                                    "E07000244","E07000245"],
}

MSA_TIERS = {
    "Greater Manchester":           "Established",
    "West Midlands":                "Established",
    "North East":                   "Established",
    "Liverpool City Region":        "Established",
    "South Yorkshire":              "Established",
    "West Yorkshire":               "Established",
    "West of England":              "Established",
    "Lancashire":                   "Established",
    "Tees Valley":                  "Existing",
    "York & North Yorkshire":       "Existing",
    "Hull & East Yorkshire":        "Existing",
    "Greater Lincolnshire":         "Existing",
    "East Midlands":                "Existing",
    "Cambridgeshire & Peterborough":"Existing",
    "Sussex & Brighton":            "DPP",
    "Hampshire & Solent":           "DPP",
    "Cheshire & Warrington":        "DPP",
    "Cumbria":                      "DPP",
    "Greater Essex":                "DPP",
    "Norfolk & Suffolk":            "DPP",
}

# Validate all LA codes exist in dataset
available = set(la_df["la_code"].tolist())
print("\n   Checking LA codes against MYE5:")
for msa, codes in MSA_LAS.items():
    missing = [c for c in codes if c not in available]
    if missing:
        print(f"   ⚠  {msa}: {len(missing)} missing — {missing}")
    else:
        print(f"   ✓  {msa}: all {len(codes)} codes found")

# ── 3. Aggregate to MSA level ─────────────────────────────────────────────
print("\n── 3. Aggregating to MSA level ──────────────────────────────────")

years = [str(y) for y in range(2011, 2025)]
pop_cols   = [f"pop_{y}"  for y in years]
avail_cols = [c for c in pop_cols if c in la_df.columns]

rows = []
for msa_name, la_codes in MSA_LAS.items():
    subset = la_df[la_df["la_code"].isin(la_codes)]
    if subset.empty:
        print(f"   ⚠  No data for {msa_name}")
        continue

    row = {"msa_name": msa_name, "tier": MSA_TIERS[msa_name]}
    row["la_count"]    = len(subset)
    row["la_matched"]  = len(subset)
    row["area_km2"]    = subset["area_km2"].sum()

    for col in avail_cols:
        row[col] = pd.to_numeric(subset[col], errors="coerce").sum()

    # Density uses 2024 pop / total area
    row["density_2024"] = round(row["pop_2024"] / row["area_km2"], 1) if row["area_km2"] > 0 else None

    # Growth rates
    if row.get("pop_2011") and row["pop_2011"] > 0:
        row["growth_2011_2024"] = round((row["pop_2024"] - row["pop_2011"]) / row["pop_2011"] * 100, 2)
    if row.get("pop_2021") and row["pop_2021"] > 0:
        row["growth_2021_2024"] = round((row["pop_2024"] - row["pop_2021"]) / row["pop_2021"] * 100, 2)

    rows.append(row)

msa_df = pd.DataFrame(rows)
print(f"   MSAs built: {len(msa_df)}")

# ── 4. England benchmarks ─────────────────────────────────────────────────
print("\n── 4. England benchmarks ────────────────────────────────────────")

eng_pop_2024    = float(england_row["pop_2024"])
eng_pop_2011    = float(england_row["pop_2011"]) if "pop_2011" in england_row else None
eng_pop_2021    = float(england_row["pop_2021"]) if "pop_2021" in england_row else None
eng_area        = float(england_row["area_km2"])
eng_density     = round(eng_pop_2024 / eng_area, 1)
eng_growth_2021 = round((eng_pop_2024 - eng_pop_2021) / eng_pop_2021 * 100, 2) if eng_pop_2021 else None
eng_growth_2011 = round((eng_pop_2024 - eng_pop_2011) / eng_pop_2011 * 100, 2) if eng_pop_2011 else None

print(f"   England population 2024:       {eng_pop_2024:>14,.0f}")
print(f"   England area (km²):            {eng_area:>14,.0f}")
print(f"   England density (per km²):     {eng_density:>14.1f}")
print(f"   England growth 2021-2024:      {eng_growth_2021:>14.2f}%")
print(f"   England growth 2011-2024:      {eng_growth_2011:>14.2f}%")

# ── 5. Rankings ───────────────────────────────────────────────────────────
print("\n── 5. Rankings (1 = largest / most dense / fastest growing) ─────")

msa_df["rank_pop_2024"]      = msa_df["pop_2024"].rank(ascending=False, method="min").astype(int)
msa_df["rank_density_2024"]  = msa_df["density_2024"].rank(ascending=False, method="min").astype(int)
msa_df["rank_growth_2021_24"]= msa_df["growth_2021_2024"].rank(ascending=False, method="min").astype(int)
msa_df["rank_growth_2011_24"]= msa_df["growth_2011_2024"].rank(ascending=False, method="min").astype(int)

# ── 6. Print results table ────────────────────────────────────────────────
print("\n── Population results (sorted by population, largest first) ─────")
print(f"\n{'MSA':<32} {'Tier':<12} {'Pop 2024':>12} {'Rank':>5} {'Density':>9} {'Rank':>5} {'Growth 21-24':>13} {'Rank':>5}")
print("-" * 105)

for _, r in msa_df.sort_values("pop_2024", ascending=False).iterrows():
    print(
        f"{r['msa_name']:<32} "
        f"{r['tier']:<12} "
        f"{r['pop_2024']:>12,.0f} "
        f"{r['rank_pop_2024']:>5} "
        f"{r['density_2024']:>9.1f} "
        f"{r['rank_density_2024']:>5} "
        f"{r['growth_2021_2024']:>13.2f}% "
        f"{r['rank_growth_2021_24']:>5}"
    )

print(f"\n{'ENGLAND (benchmark)':<32} {'':12} {eng_pop_2024:>12,.0f} {'—':>5} {eng_density:>9.1f} {'—':>5} {eng_growth_2021:>13.2f}% {'—':>5}")

# ── 7. Save to CSV for Excel update ───────────────────────────────────────
out_csv = f"{BASE}/data/processed/msa_population_ranks.csv"
msa_df.to_csv(out_csv, index=False)
print(f"\n── Saved: {out_csv}")
print("   Next step: run update_excel_pop.py to write ranks into tracker")

── 1. Reading MYE5 ──────────────────────────────────────────────
   LA rows loaded: 318
   Columns: ['la_code', 'la_name', 'geography', 'area_km2', 'pop_2024', 'density_2024', 'pop_2023', 'density_2023']
   England population 2024: 58,620,101

── 2. Defining MSA boundaries ───────────────────────────────────

   Checking LA codes against MYE5:
   ✓  Greater Manchester: all 10 codes found
   ✓  West Midlands: all 7 codes found
   ✓  North East: all 12 codes found
   ✓  Liverpool City Region: all 6 codes found
   ✓  South Yorkshire: all 4 codes found
   ✓  West Yorkshire: all 5 codes found
   ✓  West of England: all 4 codes found
   ✓  Lancashire: all 14 codes found
   ✓  Tees Valley: all 5 codes found
   ✓  York & North Yorkshire: all 2 codes found
   ✓  Hull & East Yorkshire: all 2 codes found
   ✓  Greater Lincolnshire: all 9 codes found
   ✓  East Midlands: all 23 codes found
   ✓  Cambridgeshire & Peterborough: all 6 codes found
   ✓  Sussex & Brighton: all 13 codes found
   ✓  Ham

In [8]:
"""
update_excel_pop.py
===================
Writes population ranks and values from msa_population_ranks.csv
back into the MSA Indicator Tracker Excel file.

Run from project root:
  python scripts/update_excel_pop.py
"""

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

BASE      = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"
XLSX_PATH = f"{BASE}/data/analysis/MSA_Indicator_Tracker.xlsx"
CSV_PATH  = f"{BASE}/data/processed/msa_population_ranks.csv"

# ── Load data ──────────────────────────────────────────────────────────────
ranks = pd.read_csv(CSV_PATH)
wb    = load_workbook(XLSX_PATH)

YNYCA_NAME   = "York & North Yorkshire"
ENG_POP_2024 = 58620101
ENG_DENSITY  = 449.8

ynyca = ranks[ranks["msa_name"] == YNYCA_NAME].iloc[0]

print(f"YNYCA data loaded:")
print(f"  Population 2024:     {ynyca['pop_2024']:,.0f}")
print(f"  Density 2024:        {ynyca['density_2024']:.1f}")
print(f"  Pop rank (of 20):    {ynyca['rank_pop_2024']}")
print(f"  Density rank:        {ynyca['rank_density_2024']}")
print(f"  Growth 2021-24:      {ynyca['growth_2021_2024']:.2f}%")
print(f"  Growth rank:         {ynyca['rank_growth_2021_24']}")

# ── Helper ─────────────────────────────────────────────────────────────────
GREEN_BG = "C8E6C9"
DARK     = "1A1A1A"

def write(ws, row, col, value, fmt=None, align="right", bold=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font = Font(name="Arial", size=9, bold=bold, color=DARK)
    c.alignment = Alignment(horizontal=align, vertical="center")
    if fmt:
        c.number_format = fmt
    return c

def mark_populated(ws, row):
    c = ws.cell(row=row, column=14, value="Populated")
    c.fill = PatternFill("solid", fgColor=GREEN_BG)
    c.font = Font(name="Arial", size=9)
    c.alignment = Alignment(horizontal="center", vertical="center")

# ── Find indicator rows in Sheet 2 ────────────────────────────────────────
ws2 = wb["2. YNYCA Profile"]
id_to_row = {}
for row in ws2.iter_rows(min_row=4):
    v = row[0].value
    if v:
        id_to_row[v] = row[0].row

# ── Update Sheet 3: MSA Comparison — fill all 20 MSAs ─────────────────────
ws3 = wb["3. MSA Comparison"]

# Find column positions in row 2
msa_cols = {}
for col in range(1, 30):
    v = ws3.cell(row=2, column=col).value
    if v:
        # Header is "ABBR\nFull Name" — extract full name
        parts = str(v).split("\n")
        if len(parts) >= 2:
            msa_cols[parts[1].strip()] = col
        elif len(parts) == 1:
            msa_cols[parts[0].strip()] = col

# Map full names in comparison sheet to names in ranks CSV
NAME_MAP = {
    "Greater Manchester":           "Greater Manchester",
    "West Midlands":                "West Midlands",
    "North East":                   "North East",
    "Liverpool City Region":        "Liverpool City Region",
    "South Yorkshire":              "South Yorkshire",
    "West Yorkshire":               "West Yorkshire",
    "West of England":              "West of England",
    "Lancashire":                   "Lancashire",
    "Tees Valley":                  "Tees Valley",
    "York & North Yorkshire":       "York & North Yorkshire",
    "Hull & East Yorkshire":        "Hull & East Yorkshire",
    "Greater Lincolnshire":         "Greater Lincolnshire",
    "East Midlands":                "East Midlands",
    "Cambridgeshire & Pboro":       "Cambridgeshire & Peterborough",
    "Sussex & Brighton":            "Sussex & Brighton",
    "Hampshire & Solent":           "Hampshire & Solent",
    "Cheshire & Warrington":        "Cheshire & Warrington",
    "Cumbria":                      "Cumbria",
    "Greater Essex":                "Greater Essex",
    "Norfolk & Suffolk":            "Norfolk & Suffolk",
}

# Find SNAP_001 and SNAP_002 rows in Sheet 3
comp_id_rows = {}
for row in ws3.iter_rows(min_row=3):
    v = row[0].value
    if v:
        comp_id_rows[v] = row[0].row

snap1_row = comp_id_rows.get("SNAP_001")
snap2_row = comp_id_rows.get("SNAP_002")

tier_colours = {
    "Established": "1F3864",
    "Existing":    "2A9D8F",
    "DPP":         "888888",
}

print(f"\n── Updating Sheet 3 MSA Comparison ─────────────────────────────")
for sheet_name, csv_name in NAME_MAP.items():
    col = msa_cols.get(sheet_name)
    if col is None:
        print(f"  ⚠  Column not found for: {sheet_name}")
        continue

    row_data = ranks[ranks["msa_name"] == csv_name]
    if row_data.empty:
        print(f"  ⚠  No rank data for: {csv_name}")
        continue

    r = row_data.iloc[0]
    tier = r["tier"]
    col_bg = tier_colours.get(tier, "888888")

    if snap1_row:
        c = ws3.cell(row=snap1_row, column=col, value=int(r["pop_2024"]))
        c.number_format = "#,##0"
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center", vertical="center")

    if snap2_row:
        c = ws3.cell(row=snap2_row, column=col, value=float(r["density_2024"]))
        c.number_format = "0.0"
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center", vertical="center")

    print(f"  ✓  {sheet_name}: pop={int(r['pop_2024']):,}  density={r['density_2024']:.1f}")

# Also update England avg column (col 4) on Sheet 3
if snap1_row:
    c = ws3.cell(row=snap1_row, column=4, value=ENG_POP_2024)
    c.number_format = "#,##0"
    c.font = Font(name="Arial", size=9)
    c.alignment = Alignment(horizontal="center", vertical="center")
if snap2_row:
    c = ws3.cell(row=snap2_row, column=4, value=ENG_DENSITY)
    c.number_format = "0.0"
    c.font = Font(name="Arial", size=9)
    c.alignment = Alignment(horizontal="center", vertical="center")

# ── Update Sheet 2: YNYCA Profile ─────────────────────────────────────────
print(f"\n── Updating Sheet 2 YNYCA Profile ───────────────────────────────")

# SNAP_001 — Population
r1 = id_to_row.get("SNAP_001")
if r1:
    write(ws2, r1, 4,  int(ynyca["pop_2024"]),    fmt="#,##0")
    write(ws2, r1, 6,  "Mid-2024",                align="left")
    write(ws2, r1, 7,  ENG_POP_2024,              fmt="#,##0")
    ws2.cell(row=r1, column=8).value  = f"=D{r1}-G{r1}"
    ws2.cell(row=r1, column=8).number_format = "#,##0"
    ws2.cell(row=r1, column=8).font = Font(name="Arial", size=9)
    ws2.cell(row=r1, column=8).alignment = Alignment(horizontal="right", vertical="center")
    write(ws2, r1, 9,  "N/A — size measure",      align="left")
    write(ws2, r1, 10, int(ynyca["pop_2023"]),     fmt="#,##0")
    ws2.cell(row=r1, column=11).value = f"=D{r1}-J{r1}"
    ws2.cell(row=r1, column=11).number_format = "#,##0"
    ws2.cell(row=r1, column=11).font = Font(name="Arial", size=9)
    ws2.cell(row=r1, column=11).alignment = Alignment(horizontal="right", vertical="center")
    write(ws2, r1, 13, int(ynyca["rank_pop_2024"]), align="center", bold=True)
    write(ws2, r1, 15, "ONS Mid-Year Population Estimates mid-2024. Sum York (209,301) + North Yorkshire (635,270). Rank 17 of 20 MSAs (smallest statutory CA).",
          align="left")
    mark_populated(ws2, r1)
    print(f"  ✓  SNAP_001 — pop={int(ynyca['pop_2024']):,}  rank={int(ynyca['rank_pop_2024'])}")

# SNAP_002 — Population density
r2 = id_to_row.get("SNAP_002")
if r2:
    write(ws2, r2, 4,  float(ynyca["density_2024"]),  fmt="0.0")
    write(ws2, r2, 6,  "Mid-2024",                    align="left")
    write(ws2, r2, 7,  ENG_DENSITY,                   fmt="0.0")
    ws2.cell(row=r2, column=8).value = f"=D{r2}-G{r2}"
    ws2.cell(row=r2, column=8).number_format = "0.0"
    ws2.cell(row=r2, column=8).font = Font(name="Arial", size=9)
    ws2.cell(row=r2, column=8).alignment = Alignment(horizontal="right", vertical="center")
    write(ws2, r2, 9,  "N/A — context measure",       align="left")
    ynyca_density_2023 = round(float(ynyca["pop_2023"]) / float(ynyca["area_km2"]), 1)
    write(ws2, r2, 10, ynyca_density_2023,             fmt="0.0")
    ws2.cell(row=r2, column=11).value = f"=D{r2}-J{r2}"
    ws2.cell(row=r2, column=11).number_format = "0.0"
    ws2.cell(row=r2, column=11).font = Font(name="Arial", size=9)
    ws2.cell(row=r2, column=11).alignment = Alignment(horizontal="right", vertical="center")
    write(ws2, r2, 13, int(ynyca["rank_density_2024"]), align="center", bold=True)
    write(ws2, r2, 15, f"Calculated: {int(ynyca['pop_2024']):,} pop / {int(ynyca['area_km2']):,} km². "
          f"Rank 19 of 20 — second least dense MSA after Cumbria ({ranks[ranks['msa_name']=='Cumbria']['density_2024'].values[0]:.1f}/km²). "
          f"England avg 449.8/km².", align="left")
    mark_populated(ws2, r2)
    print(f"  ✓  SNAP_002 — density={float(ynyca['density_2024']):.1f}/km²  rank={int(ynyca['rank_density_2024'])}")

# ── Save ───────────────────────────────────────────────────────────────────
wb.save(XLSX_PATH)
print(f"\n── Saved: {XLSX_PATH}")
print("\n── Key findings for YNYCA profile ──────────────────────────────")
print(f"  Population:     {int(ynyca['pop_2024']):,}  (rank {int(ynyca['rank_pop_2024'])}/20 — smallest statutory CA)")
print(f"  Density:        {float(ynyca['density_2024']):.1f}/km²  (rank {int(ynyca['rank_density_2024'])}/20 — second least dense)")
print(f"  Growth 21-24:   {float(ynyca['growth_2021_2024']):.2f}%  (rank {int(ynyca['rank_growth_2021_24'])}/20 — below England avg {3.65}%)")
print(f"  Growth 11-24:   {float(ynyca['growth_2011_2024']):.2f}%  (rank to be confirmed)")

YNYCA data loaded:
  Population 2024:     844,571
  Density 2024:        101.6
  Pop rank (of 20):    17
  Density rank:        19
  Growth 2021-24:      2.94%
  Growth rank:         17

── Updating Sheet 3 MSA Comparison ─────────────────────────────
  ✓  Greater Manchester: pop=3,009,664  density=2358.7
  ✓  West Midlands: pop=3,036,605  density=3367.6
  ✓  North East: pop=2,760,678  density=321.7
  ✓  Liverpool City Region: pop=1,607,084  density=2197.6
  ✓  South Yorkshire: pop=1,430,623  density=922.1
  ✓  West Yorkshire: pop=2,435,236  density=1200.1
  ✓  West of England: pop=1,225,337  density=923.8
  ✓  Lancashire: pop=1,601,645  density=522.4
  ✓  Tees Valley: pop=712,858  density=896.6
  ✓  York & North Yorkshire: pop=844,571  density=101.6
  ✓  Hull & East Yorkshire: pop=631,285  density=255.0
  ✓  Greater Lincolnshire: pop=1,120,749  density=160.6
  ✓  East Midlands: pop=3,400,821  density=460.8
  ✓  Cambridgeshire & Pboro: pop=933,972  density=275.5
  ✓  Sussex & Brighton: